# Baseline Models: TF-IDF + MNB / Logistic Regression / Linear SVM

Train and evaluate three classical baselines on the KazSAnDRA polarity classification task.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.data_loader import load_kazsandra
from src.baselines import build_mnb_pipeline, build_logreg_pipeline, build_svm_pipeline
from src.evaluation import compute_metrics, print_report, plot_confusion_matrix, plot_roc_curve, save_metrics

sns.set_theme(style="whitegrid")
print("Imports done.")

## 1. Load Data

In [ ]:
splits = load_kazsandra()
train_df = splits["train"]
val_df = splits["val"]
test_df = splits["test"]

X_train = train_df["text_cleaned"].fillna("")
y_train = train_df["label"]
X_val = val_df["text_cleaned"].fillna("")
y_val = val_df["label"]
X_test = test_df["text_cleaned"].fillna("")
y_test = test_df["label"]

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")
print(f"Positive %: train={y_train.mean()*100:.1f}%  val={y_val.mean()*100:.1f}%  test={y_test.mean()*100:.1f}%")

## 2. Train Multinomial Naive Bayes

In [ ]:
mnb = build_mnb_pipeline()

start = time.time()
mnb.fit(X_train, y_train)
train_time_mnb = time.time() - start
print(f"Multinomial Naive Bayes trained in {train_time_mnb:.1f}s")

y_val_pred_mnb = mnb.predict(X_val)
y_val_prob_mnb = mnb.predict_proba(X_val)[:, 1]

print("\n--- Validation Set Results ---")
print_report(y_val, y_val_pred_mnb)
val_metrics_mnb = compute_metrics(y_val, y_val_pred_mnb, y_val_prob_mnb)
print(f"ROC-AUC: {val_metrics_mnb['roc_auc']:.4f}")

## 3. Train Logistic Regression

In [ ]:
logreg = build_logreg_pipeline()

start = time.time()
logreg.fit(X_train, y_train)
train_time_lr = time.time() - start
print(f"Logistic Regression trained in {train_time_lr:.1f}s")

y_val_pred_lr = logreg.predict(X_val)
y_val_prob_lr = logreg.predict_proba(X_val)[:, 1]

print("\n--- Validation Set Results ---")
print_report(y_val, y_val_pred_lr)
val_metrics_lr = compute_metrics(y_val, y_val_pred_lr, y_val_prob_lr)
print(f"ROC-AUC: {val_metrics_lr['roc_auc']:.4f}")

## 4. Train Linear SVM

In [ ]:
svm = build_svm_pipeline()

start = time.time()
svm.fit(X_train, y_train)
train_time_svm = time.time() - start
print(f"Linear SVM trained in {train_time_svm:.1f}s")

y_val_pred_svm = svm.predict(X_val)
y_val_prob_svm = svm.predict_proba(X_val)[:, 1]

print("\n--- Validation Set Results ---")
print_report(y_val, y_val_pred_svm)
val_metrics_svm = compute_metrics(y_val, y_val_pred_svm, y_val_prob_svm)
print(f"ROC-AUC: {val_metrics_svm['roc_auc']:.4f}")

## 5. Test Set Evaluation

In [ ]:
# MNB on test set
y_test_pred_mnb = mnb.predict(X_test)
y_test_prob_mnb = mnb.predict_proba(X_test)[:, 1]
test_metrics_mnb = compute_metrics(y_test, y_test_pred_mnb, y_test_prob_mnb)

# LogReg on test set
y_test_pred_lr = logreg.predict(X_test)
y_test_prob_lr = logreg.predict_proba(X_test)[:, 1]
test_metrics_lr = compute_metrics(y_test, y_test_pred_lr, y_test_prob_lr)

# SVM on test set
y_test_pred_svm = svm.predict(X_test)
y_test_prob_svm = svm.predict_proba(X_test)[:, 1]
test_metrics_svm = compute_metrics(y_test, y_test_pred_svm, y_test_prob_svm)

print("=" * 60)
print("MULTINOMIAL NAIVE BAYES — Test Set")
print("=" * 60)
print_report(y_test, y_test_pred_mnb)
print(f"ROC-AUC: {test_metrics_mnb['roc_auc']:.4f}\n")

print("=" * 60)
print("LOGISTIC REGRESSION — Test Set")
print("=" * 60)
print_report(y_test, y_test_pred_lr)
print(f"ROC-AUC: {test_metrics_lr['roc_auc']:.4f}\n")

print("=" * 60)
print("LINEAR SVM — Test Set")
print("=" * 60)
print_report(y_test, y_test_pred_svm)
print(f"ROC-AUC: {test_metrics_svm['roc_auc']:.4f}")

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_confusion_matrix(y_test, y_test_pred_mnb, title="Multinomial NB", ax=axes[0])
plot_confusion_matrix(y_test, y_test_pred_lr, title="Logistic Regression", ax=axes[1])
plot_confusion_matrix(y_test, y_test_pred_svm, title="Linear SVM", ax=axes[2])

plt.tight_layout()
plt.show()

## 7. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

plot_roc_curve(y_test, y_test_prob_mnb, label="Multinomial NB", ax=ax)
plot_roc_curve(y_test, y_test_prob_lr, label="Logistic Regression", ax=ax)
plot_roc_curve(y_test, y_test_prob_svm, label="Linear SVM", ax=ax)

plt.tight_layout()
plt.show()

## 8. Results Comparison

In [ ]:
comparison = pd.DataFrame({
    "Multinomial NB": test_metrics_mnb,
    "Logistic Regression": test_metrics_lr,
    "Linear SVM": test_metrics_svm,
}).round(4)

comparison

## 9. Save Metrics & Predictions

In [ ]:
all_metrics = {
    "multinomial_nb": {
        "val": val_metrics_mnb,
        "test": test_metrics_mnb,
        "train_time_seconds": round(train_time_mnb, 2),
        "hyperparameters": {
            "max_features": 50000,
            "ngram_range": [1, 2],
            "sublinear_tf": True,
            "alpha": 1.0,
        },
    },
    "logistic_regression": {
        "val": val_metrics_lr,
        "test": test_metrics_lr,
        "train_time_seconds": round(train_time_lr, 2),
        "hyperparameters": {
            "max_features": 50000,
            "ngram_range": [1, 2],
            "C": 1.0,
            "solver": "lbfgs",
            "sublinear_tf": True,
        },
    },
    "linear_svm": {
        "val": val_metrics_svm,
        "test": test_metrics_svm,
        "train_time_seconds": round(train_time_svm, 2),
        "hyperparameters": {
            "max_features": 50000,
            "ngram_range": [1, 2],
            "C": 1.0,
            "sublinear_tf": True,
            "calibration_cv": 3,
        },
    },
}

save_metrics(all_metrics, "../results/baseline_metrics.json")

# Save test predictions for error analysis
pred_df = test_df[["custom_id", "text_cleaned", "label", "domain"]].copy()
pred_df["mnb_pred"] = y_test_pred_mnb
pred_df["mnb_prob"] = y_test_prob_mnb
pred_df["logreg_pred"] = y_test_pred_lr
pred_df["logreg_prob"] = y_test_prob_lr
pred_df["svm_pred"] = y_test_pred_svm
pred_df["svm_prob"] = y_test_prob_svm
os.makedirs("../results", exist_ok=True)
pred_df.to_csv("../results/baseline_predictions.csv", index=False)
print(f"Predictions saved ({len(pred_df)} rows)")
print("Done!")